# KEEP occurrence-rule ablation

Question: with the same diagnoses, roll-up targets and concept axes, how do one/two/three diagnosis rows and two distinct admissions change patient presence and co-occurrence?

This synthetic study extends the [ancestor-rollup variants](keep_ancestor_rollup_variants.ipynb). Baseline selection and ancestor correctness belong there. Here the fixed project profile is dense `all_retained`, direct traversal, depth five, root exclusion, and post-roll-up thresholding. All histories are invented. The default remains two distinct diagnoses.

We compare four arms, assert an independent hand-derived oracle before interpreting measurements, then reconcile patient contributions to count changes. No Athena, MIMIC, private config, or embedding training is required.


In [1]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="`torch.jit.script` is deprecated.*", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from examples.keep_occurrence_rule_ablation import RULES, run_study
import torch
torch.sparse.check_sparse_tensor_invariants.enable()
from examples.keep_ancestor_rollup_ablation_utils import trace_pair
report = run_study()
assert report["hand_oracle_passed"] and report["public_counter_agreement"]
display(pd.DataFrame(RULES).T)
print("Passed: literal presence/count oracle and public OMOP / mapped-MIMIC / independent counter agreement.")


,min_occurrences,occurrence_unit
rows_2,2,diagnosis
rows_1,1,diagnosis
rows_3,3,diagnosis
admissions_2,2,admission


Passed: literal presence/count oracle and public OMOP / mapped-MIMIC / independent counter agreement.


## Contract and controls

Thresholds apply **after** mapping and roll-up. Union targets within each diagnosis. Exact diagnosis copies collapse; conflicting copies fail. Two codes or paths reaching a target from one diagnosis supply one diagnosis of support.

Admission mode unions target sets within each patient's admission. Three diagnoses in one admission are three rows but one admission; admission IDs are patient-scoped. Mapped diagnoses missing admission IDs raise rather than becoming artificial admissions. Our invented visit IDs explicitly represent admissions; generic OMOP encounters require admission-level curation first.

Keep all ten patients, including empty and unmapped histories. Row thresholds are nested (3 ⊆ 2 ⊆ 1); two-admission presence is a subset of two-row presence. Three rows and two admissions need not be nested.


In [2]:
traces = report["traces"]
display(pd.DataFrame([
    {"patient": row["patient_id"], "row_support": row["row_support"],
     "admission_support": row["admission_support"],
     **{arm: traces[arm][i]["present"] for arm in RULES}}
    for i, row in enumerate(traces["rows_2"])
]))


,patient,row_support,admission_support,rows_2,rows_1,rows_3,admissions_2
0,p1,"{'101': 1, '102': 1, '103': 1, '104': 1, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'...",[],"[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[],[]
1,p2,"{'101': 2, '102': 2, '103': 2, '104': 2, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[],[]
2,p3,"{'101': 2, '102': 2, '103': 2, '104': 2, '105'...","{'101': 2, '102': 2, '103': 2, '104': 2, '105'...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[],"[101, 102, 103, 104, 105, 201, 202, 203, 204, ..."
3,p4,"{'101': 3, '102': 3, '103': 3, '104': 3, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[]
4,p5,"{'101': 3, '102': 3, '103': 3, '104': 3, '105'...","{'101': 3, '102': 3, '103': 3, '104': 3, '105'...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ...","[101, 102, 103, 104, 105, 201, 202, 203, 204, ..."
5,p6,"{'101': 1, '102': 1, '103': 1, '104': 1, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'...",[],"[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[],[]
6,p7,{},{},[],[],[],[]
7,p8,{},{},[],[],[],[]
8,p9,"{'105': 1, '205': 1}","{'105': 1, '205': 1}",[],"[105, 205]",[],[]
9,p10,"{'101': 1, '102': 1, '103': 1, '104': 1, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'...",[105],"[101, 102, 103, 104, 105, 201, 202, 203, 204, ...",[],[105]


## Aggregate outcomes

Graph and axes are fixed; this intervention changes patient presence, not graph size. Diagonals measure prevalence; positive unordered off-diagonal pairs measure support. Report the common possible-pair denominator alongside density and concentration. Normalize pair-mass entropy using that fixed denominator.


In [3]:
aggregates = pd.DataFrame(report["aggregates"]).T
display(aggregates[["patients", "patients_with_present_targets", "conditions_present",
                    "observed_pairs", "possible_pairs", "pair_density", "pair_mass",
                    "normalized_entropy"]].round(5))
comparisons = report["comparisons"]
summary = pd.DataFrame({arm: result["summary"] for arm, result in comparisons.items()}).T
display(summary[["changed_count_pairs", "observed_pair_union", "changed_pair_fraction",
                 "gained_pairs", "lost_pairs", "absolute_count_delta",
                 "global_pair_mass_jsd"]].round(5))


,patients,patients_with_present_targets,conditions_present,observed_pairs,possible_pairs,pair_density,pair_mass,normalized_entropy
rows_2,10,5,10,45,55,0.818182,180,0.949924
rows_1,10,8,10,45,55,0.818182,316,0.949871
rows_3,10,2,10,45,55,0.818182,90,0.949924
admissions_2,10,3,10,45,55,0.818182,90,0.949924


,changed_count_pairs,observed_pair_union,changed_pair_fraction,gained_pairs,lost_pairs,absolute_count_delta,global_pair_mass_jsd
rows_1,45.0,45.0,1.0,0.0,0.0,136.0,0.00007
rows_3,45.0,45.0,1.0,0.0,0.0,90.0,0.00000
admissions_2,45.0,45.0,1.0,0.0,0.0,90.0,0.00000


## Explain a lost pair

Patient p2 has two deep diagnoses in one admission; p3 has two in separate admissions. Both satisfy two rows; only p3 satisfies two admissions. The trace reconciles every changed patient contribution with the public matrix delta. Empty patients remain in N throughout.


In [4]:
pair = ("105", "205")
witnesses = trace_pair(traces["rows_2"], traces["admissions_2"], *pair)
row = next(r for r in comparisons["admissions_2"]["pairs"] if (r["i"], r["j"]) == pair)
assert sum(w["pair_delta"] for w in witnesses) == row["count_delta"]
display(pd.DataFrame([
    {"patient": w["patient_id"], "delta": w["pair_delta"],
     "row_support": w["before"]["row_support"],
     "admission_support": w["after"]["admission_support"]}
    for w in witnesses
]))
print("Pair:", pair, "before:", row["before"]["count"], "after:", row["after"]["count"])


,patient,delta,row_support,admission_support
0,p2,-1,"{'101': 2, '102': 2, '103': 2, '104': 2, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'..."
1,p4,-1,"{'101': 3, '102': 3, '103': 3, '104': 3, '105'...","{'101': 1, '102': 1, '103': 1, '104': 1, '105'..."


Pair: ('105', '205') before: 4 after: 2


## Association and distribution measures

Reuse the ancestor-rollup statistics: patient PPMI/NPMI, both conditional probabilities, fixed GloVe log targets/weights, per-concept cosine/JSD, global pair-mass JSD, normalized entropy and concentration. These describe preprocessing consequences, not clinical benefit. PPMI can change through marginals even when a pair count stays fixed. Keep undefined values and raw support visible; no smoothing or rare-pair filtering changes counts. NPMI is undefined for unobserved pairs and joint prevalence one; empty profiles/distributions also remain undefined.


In [5]:
selected = comparisons["admissions_2"]
pairs = pd.json_normalize(selected["pairs"])
display(pairs[["i", "j", "before.count", "after.count", "count_delta",
               "before.ppmi", "after.ppmi", "before.npmi", "after.npmi"]].round(5))
display(pd.DataFrame(selected["profiles"]))
print("Input digest:", report["histories_sha256"])
print("Completed four synthetic arms and three baseline comparisons.")


,i,j,before.count,after.count,count_delta,before.ppmi,after.ppmi,before.npmi,after.npmi
0,101,102,4,2,-2,0.91629,1.60944,1.00000,1.00000
1,101,103,4,2,-2,0.91629,1.60944,1.00000,1.00000
2,101,104,4,2,-2,0.91629,1.60944,1.00000,1.00000
3,101,105,4,2,-2,0.69315,1.20397,0.75647,0.74807
4,101,201,4,2,-2,0.91629,1.60944,1.00000,1.00000
5,101,202,4,2,-2,0.91629,1.60944,1.00000,1.00000
6,101,203,4,2,-2,0.91629,1.60944,1.00000,1.00000
7,101,204,4,2,-2,0.91629,1.60944,1.00000,1.00000
8,101,205,4,2,-2,0.91629,1.60944,1.00000,1.00000
9,102,103,4,2,-2,0.91629,1.60944,1.00000,1.00000


,concept_id,count_cosine,count_jsd,ppmi_cosine,ppmi_jsd
0,101,1.0,0.0,0.999996,1.766710e-06
1,102,1.0,0.0,0.999996,1.766710e-06
2,103,1.0,0.0,0.999996,1.766710e-06
3,104,1.0,0.0,0.999996,1.766710e-06
4,105,1.0,0.0,1.000000,1.601713e-16
5,201,1.0,0.0,0.999996,1.766710e-06
6,202,1.0,0.0,0.999996,1.766710e-06
7,203,1.0,0.0,0.999996,1.766710e-06
8,204,1.0,0.0,0.999996,1.766710e-06
9,205,1.0,0.0,0.999996,1.766710e-06


Input digest: 5ac96528e7557c32a364c73c0f98f78f076ecc48a4941d6ee073bdf88a03dea3
Completed four synthetic arms and three baseline comparisons.


## Repeat and extend

From the repository root in the same Python environment:

```sh
python -m examples.keep_occurrence_rule_ablation --output /tmp/keep-occurrence-new
python -m unittest discover -s tests/core -p test_keep_occurrence_rule.py -v
```

Choose a new output directory. The JSON report records settings, input/source hashes, matrices, aggregates and patient traces. The public mapped-MIMIC route here starts from mapped history records; reader/ICD integration remains covered separately. Real cohort comparisons need the same authorized patients and admission identities across arms. This notebook does not discover those inputs or establish population effects, author-artifact parity, or improved embeddings.
